In [ ]:
%load_ext memprofiler
import dask.array as da
import numpy as np
import zarr
from numcodecs import Blosc

import blosc2

In [ ]:
N = 70_000

# For best speed
# blosc2.cparams_dflts["codec"] = blosc2.Codec.BLOSCLZ
blosc2.cparams_dflts["codec"] = blosc2.Codec.LZ4
blosc2.cparams_dflts["clevel"] = 1
# compressor = Blosc(cname='blosclz', clevel=1, shuffle=Blosc.SHUFFLE)
compressor = Blosc(cname="lz4", clevel=1, shuffle=Blosc.SHUFFLE)

In [ ]:
%%time
na = np.linspace(0, 1, N * N).reshape(N, N)
a = blosc2.asarray(na)
za = zarr.array(na, compressor=compressor, zarr_format=2, chunks=a.chunks)
del na
nb = np.linspace(1, 2, N * N).reshape(N, N)
b = blosc2.asarray(nb)
zb = zarr.array(nb, compressor=compressor, zarr_format=2, chunks=b.chunks)
del nb
nc = np.linspace(-10, 10, N * N).reshape(N, N)
c = blosc2.asarray(nc)
zc = zarr.array(nc, compressor=compressor, zarr_format=2, chunks=c.chunks)
del nc

In [ ]:
%%time
# Expression (blosc2 form)
# expr = (a * 2 + b > c)
# expr = ((a ** 3 + blosc2.sin(c * 2)) < b)
expr = ((a**3 + blosc2.sin(c * 2)) < b) & (c > 0)

In [ ]:
%%mprof_run 1.LazyArray::compute-LZ4-1
# Evaluate and get a NDArray as result
out = expr.compute()

In [ ]:
out.info

In [ ]:
%%mprof_run 2.LazyArray::getitem-LZ4-1
# Evaluate and get a NDArray as result
out_ = expr[:]

In [ ]:
%%time
# Expression (dask form)
da_ = da.from_zarr(za)
db = da.from_zarr(zb)
dc = da.from_zarr(zc)
# dexpr = (da_ * 2 + db > dc)
# dexpr = ((da_ ** 3 + da.sin(dc * 2)) < db)
dexpr = ((da_**3 + da.sin(dc * 2)) < db) & (dc > 0)
scheduler = "single-threaded" if blosc2.nthreads == 1 else "threads"

In [ ]:
%%mprof_run 3.Dask::to_zarr-LZ4-1
zres = zarr.open(shape=(N, N), dtype=dexpr.dtype, compressor=compressor, zarr_format=2, chunks=a.chunks)
#with dask.config.set(scheduler=scheduler):
with dask.config.set(scheduler=scheduler, num_workers=blosc2.nthreads):
    da.to_zarr(dexpr, zres)

In [ ]:
%%mprof_run 4.Dask::compute-LZ4-1
#with dask.config.set(scheduler=scheduler):
with dask.config.set(scheduler=scheduler, num_workers=blosc2.nthreads):
    nres = dexpr.compute()

In [ ]:
%mprof_plot .* -t "AMD 9800X3D -- Number of threads: {blosc2.nthreads}"